# Notebook 06: Gymnasium Environment Demo

This notebook demonstrates the `SovereignRiskEnv` Gymnasium environment built in Phase 3.
It covers:

1. **Instantiate and inspect** — create all 9 profiles, print spaces and initial states.
2. **Random agent episode** — run a full 30-step episode on `Developing_High` with a random policy.
3. **Policy strategy comparison** — compare austerity vs. maintain vs. stimulus on `Advanced_Low`.
4. **Visualise one episode** — 4 publication-quality plots for a single `Developing_High` episode.
5. **Profile comparison** — box plots of terminal debt across all 9 profiles with passive policy.

In [ ]:
import sys
import os

# Ensure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from src.environment.sovereign_risk_env import SovereignRiskEnv
from src.environment.config import list_profiles

# Path configuration
CONFIG_PATH  = '../data/processed/transition_parameters.json'
SCALING_PATH = '../data/processed/scaling_parameters.json'
FIGURES_DIR  = '../outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

def make_env(profile, **kwargs):
    return SovereignRiskEnv(
        profile=profile,
        config_path=CONFIG_PATH,
        scaling_path=SCALING_PATH,
        **kwargs
    )

print('Environment module loaded successfully.')
print(f'Available profiles: {list_profiles(CONFIG_PATH)}')

---
## Part 8a: Instantiate and Inspect All 9 Profiles

In [ ]:
STATE_VARS = [
    'output_growth', 'debt_to_gdp', 'primary_balance',
    'interest_rate', 'climate_shock', 'adaptation_capital', 'risk_premium'
]

print(f"{'Profile':<30} {'Obs Space':<18} {'Act Space':<12} {'Init Debt':>10} {'Init Growth':>12}")
print("-" * 82)

for profile in list_profiles(CONFIG_PATH):
    env = make_env(profile, seed=42)
    obs, info = env.reset(seed=42)
    raw = info['raw_state']
    print(
        f"{profile:<30} "
        f"{str(env.observation_space.shape):<18} "
        f"Discrete({env.action_space.n})  "
        f"{raw['debt']:>10.2f}% "
        f"{raw['growth']:>11.2f}%"
    )
    env.close()

print()
print('Action space definitions:')
env_sample = make_env('Advanced_Low')
for k, (name, change) in env_sample.ACTION_MAP.items():
    print(f"  Action {k}: {name:<35} (PB change: {change:+.1f}pp of GDP)")
env_sample.close()

In [ ]:
# Print full initial state (normalised and raw) for each profile
print(f"{'Profile':<28}  {'Growth':>8} {'Debt':>8} {'PrimBal':>8} {'Rate':>8} {'Climate':>8} {'Adapt':>8} {'r-g':>8}")
print("  (raw values in % of GDP or index units)")
print("-" * 96)

for profile in list_profiles(CONFIG_PATH):
    env = make_env(profile, seed=42)
    _, info = env.reset(seed=42)
    r = info['raw_state']
    print(
        f"{profile:<28}  "
        f"{r['growth']:>8.2f} "
        f"{r['debt']:>8.1f} "
        f"{r['primary_balance']:>8.3f} "
        f"{r['interest_rate']:>8.3f} "
        f"{r['climate_damage']:>8.4f} "
        f"{r['adaptation_capital']:>8.4f} "
        f"{r['risk_premium']:>8.3f}"
    )
    env.close()

---
## Part 8b: Random Agent Episode on Developing_High

In [ ]:
np.random.seed(2024)
env = make_env('Developing_High', seed=2024, max_steps=30)
obs, info = env.reset(seed=2024)

history = []
for year in range(30):
    action = np.random.randint(0, 6)
    obs, reward, terminated, truncated, info = env.step(action)
    row = {
        'Year': 2025 + year,
        'Action': f"{action}: {info['action_name']}",
        'Growth (%)': info['raw_state']['growth'],
        'Debt (% GDP)': info['raw_state']['debt'],
        'Primary Balance': info['raw_state']['primary_balance'],
        'Climate Damage (%)': info['raw_state']['climate_damage'],
        'Reward': reward,
        **{f'R:{k}': v for k, v in info['reward_components'].items()},
    }
    history.append(row)
    if terminated or truncated:
        end_reason = 'DEBT CRISIS' if terminated else 'MAX STEPS'
        break

env.close()
df = pd.DataFrame(history)

print(f"Episode ended: {end_reason} (after {len(history)} years)")
print()

display_cols = ['Year', 'Action', 'Growth (%)', 'Debt (% GDP)', 'Primary Balance', 'Climate Damage (%)', 'Reward']
with pd.option_context('display.max_rows', 35, 'display.float_format', '{:.3f}'.format, 'display.max_colwidth', 35):
    print(df[display_cols].to_string(index=False))

print(f"\nSummary: Mean reward={df['Reward'].mean():.2f}, "
      f"Final debt={df['Debt (% GDP)'].iloc[-1]:.1f}%, "
      f"Climate events={int((df['Climate Damage (%)']>0).sum())}")

---
## Part 8c: Policy Strategy Comparison on Advanced_Low

In [ ]:
def run_policy(profile, fixed_action, n_episodes=100, max_steps=30, base_seed=0):
    """Run n_episodes with a fixed action, return summary statistics."""
    terminal_debts = []
    cumulative_rewards = []
    crisis_count = 0

    env = make_env(profile, max_steps=max_steps)

    for ep in range(n_episodes):
        obs, info = env.reset(seed=base_seed + ep)
        total_reward = 0.0
        episode_terminated = False

        for _ in range(max_steps):
            obs, reward, terminated, truncated, info = env.step(fixed_action)
            total_reward += reward
            if terminated:
                crisis_count += 1
                episode_terminated = True
            if terminated or truncated:
                break

        terminal_debts.append(info['raw_state']['debt'])
        cumulative_rewards.append(total_reward)

    env.close()

    return {
        'mean_terminal_debt': np.mean(terminal_debts),
        'median_terminal_debt': np.median(terminal_debts),
        'mean_cumulative_reward': np.mean(cumulative_rewards),
        'pct_crisis': 100 * crisis_count / n_episodes,
    }


strategies = [
    (0, 'Always Austerity (Action 0)'),
    (2, 'Maintain Policy (Action 2)'),
    (4, 'Always Stimulus (Action 4)'),
]

print("Running 100 episodes per strategy on Advanced_Low...")
results = []
for action, label in strategies:
    stats = run_policy('Advanced_Low', fixed_action=action, n_episodes=100)
    results.append({'Strategy': label, **stats})
    print(f"  Done: {label}")

df_compare = pd.DataFrame(results)
df_compare.columns = ['Strategy', 'Mean Terminal Debt (%GDP)',
                       'Median Terminal Debt (%GDP)',
                       'Mean Cumulative Reward', 'Crisis Rate (%)']
print()
with pd.option_context('display.float_format', '{:.2f}'.format):
    print(df_compare.to_string(index=False))

print()
print("Interpretation:")
print("  - Austerity should produce the lowest mean debt and fewest crises.")
print("  - Stimulus should produce highest debt and most crises.")
print("  - The environment responds sensibly to different policies.")

---
## Part 8d: Visualise One Episode — 4 Publication-Quality Plots

In [ ]:
# Run one deterministic episode on Developing_High with a random policy
np.random.seed(1234)
env = make_env('Developing_High', seed=1234, max_steps=30)
obs, info = env.reset(seed=1234)

viz_history = []
for year in range(30):
    action = np.random.randint(0, 6)
    obs, reward, terminated, truncated, info = env.step(action)
    viz_history.append({
        'year': year + 1,
        'action': action,
        'action_name': info['action_name'],
        'growth': info['raw_state']['growth'],
        'debt': info['raw_state']['debt'],
        'primary_balance': info['raw_state']['primary_balance'],
        'interest_rate': info['raw_state']['interest_rate'],
        'climate_damage': info['raw_state']['climate_damage'],
        'adaptation_capital': info['raw_state']['adaptation_capital'],
        'risk_premium': info['raw_state']['risk_premium'],
        'reward': reward,
        **info['reward_components'],
    })
    if terminated or truncated:
        break

env.close()
h = pd.DataFrame(viz_history)

print(f'Episode length: {len(h)} years')

In [ ]:
# --------------------------------------------------------------------------
# Figure 1: Debt trajectory
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(h['year'], h['debt'], color='#1f77b4', linewidth=2.5, label='Debt-to-GDP')
ax.fill_between(h['year'], h['debt'], alpha=0.12, color='#1f77b4')
ax.axhline(200, color='red',   linestyle='--', linewidth=1.5, label='Crisis threshold (200%)')
ax.axhline(60,  color='green', linestyle='--', linewidth=1.5, label='Maastricht criterion (60%)')

ax.set_xlabel('Year of Episode', fontsize=12)
ax.set_ylabel('Debt-to-GDP (%)', fontsize=12)
ax.set_title('Debt Trajectory — Developing_High Profile (Random Policy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, len(h))

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_env_debt_trajectory.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_env_debt_trajectory')

In [ ]:
# --------------------------------------------------------------------------
# Figure 2: Growth and climate shocks (dual axis)
# --------------------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(h['year'], h['growth'], color='#2ca02c', linewidth=2.5, label='GDP growth (%)')
ax1.axhline(0, color='black', linewidth=0.8, linestyle=':')

# Only plot climate bars where damage > 0
climate_years   = h[h['climate_damage'] > 0]['year']
climate_damages = h[h['climate_damage'] > 0]['climate_damage']
ax2.bar(climate_years, climate_damages, color='#d62728', alpha=0.6, width=0.6, label='Climate damage (% GDP)')

ax1.set_xlabel('Year of Episode', fontsize=12)
ax1.set_ylabel('Real GDP Growth (%)', color='#2ca02c', fontsize=12)
ax2.set_ylabel('Climate Damage (% of GDP)', color='#d62728', fontsize=12)
ax1.set_title('GDP Growth and Climate Shocks — Developing_High', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, len(h))

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_env_growth_climate.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_env_growth_climate')

In [ ]:
# --------------------------------------------------------------------------
# Figure 3: Reward decomposition (stacked bar)
# --------------------------------------------------------------------------
reward_cols = [
    'debt_penalty', 'interest_penalty', 'adjustment_penalty',
    'output_penalty', 'climate_penalty', 'reduction_reward', 'growth_reward'
]
reward_labels = [
    'Debt penalty', 'Interest burden', 'Adjustment cost',
    'Output instability', 'Climate damage', 'Debt reduction', 'Growth reward'
]
colours_neg = ['#d62728', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2']
colours_pos = ['#2ca02c', '#1f77b4']
all_colours = colours_neg + colours_pos

fig, ax = plt.subplots(figsize=(12, 5))

years = h['year'].values
pos_bottom = np.zeros(len(h))
neg_bottom = np.zeros(len(h))

for col, label, colour in zip(reward_cols, reward_labels, all_colours):
    vals = h[col].values
    pos_vals = np.where(vals > 0, vals, 0)
    neg_vals = np.where(vals < 0, vals, 0)
    ax.bar(years, pos_vals, bottom=pos_bottom, color=colour, alpha=0.85, label=label, width=0.8)
    ax.bar(years, neg_vals, bottom=neg_bottom, color=colour, alpha=0.85, width=0.8)
    pos_bottom += pos_vals
    neg_bottom += neg_vals

ax.plot(years, h['reward'].values, 'k-', linewidth=1.5, label='Total reward', zorder=5)
ax.axhline(0, color='black', linewidth=0.8)

ax.set_xlabel('Year of Episode', fontsize=12)
ax.set_ylabel('Reward (utils)', fontsize=12)
ax.set_title('Reward Decomposition — Developing_High (Random Policy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, ncol=4, loc='lower left')
ax.grid(True, alpha=0.2, axis='y')
ax.set_xlim(0.5, len(h) + 0.5)

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_env_reward_decomp.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_env_reward_decomp')

In [ ]:
# --------------------------------------------------------------------------
# Figure 4: Action sequence (colour-coded by type)
# --------------------------------------------------------------------------
action_colours = {
    0: '#d62728',   # Severe austerity — red
    1: '#ff7f0e',   # Moderate austerity — orange
    2: '#7f7f7f',   # Maintain — grey
    3: '#17becf',   # Moderate stimulus — teal
    4: '#1f77b4',   # Large stimulus — blue
    5: '#2ca02c',   # Climate adaptation — green
}
action_names_short = {
    0: 'Severe austerity',
    1: 'Moderate austerity',
    2: 'Maintain policy',
    3: 'Moderate stimulus',
    4: 'Large stimulus',
    5: 'Climate adaptation',
}

fig, ax = plt.subplots(figsize=(12, 3.5))

bar_colours = [action_colours[a] for a in h['action']]
ax.bar(h['year'], np.ones(len(h)), color=bar_colours, width=0.9, edgecolor='white', linewidth=0.5)

# Add action number labels
for _, row in h.iterrows():
    ax.text(row['year'], 0.5, str(int(row['action'])),
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Legend patches
patches = [mpatches.Patch(color=action_colours[i], label=f"{i}: {action_names_short[i]}")
           for i in range(6)]
ax.legend(handles=patches, fontsize=8, ncol=3, loc='upper right',
          bbox_to_anchor=(1.0, 1.35))

ax.set_xlabel('Year of Episode', fontsize=12)
ax.set_yticks([])
ax.set_xlim(0.5, len(h) + 0.5)
ax.set_title('Fiscal Policy Actions — Developing_High (Random Policy)', fontsize=13, fontweight='bold')
ax.grid(False)

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_env_action_sequence.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_env_action_sequence')

---
## Part 8e: Profile Comparison — Passive Policy Across All 9 Profiles

In [ ]:
print('Running 100 episodes per profile (action 2: maintain policy)...')

profile_results = {}
for profile in list_profiles(CONFIG_PATH):
    terminal_debts = []
    env = make_env(profile, max_steps=30)
    for ep in range(100):
        obs, info = env.reset(seed=ep)
        for _ in range(30):
            obs, _, terminated, truncated, info = env.step(2)
            if terminated or truncated:
                break
        terminal_debts.append(info['raw_state']['debt'])
    env.close()
    profile_results[profile] = terminal_debts
    print(f'  {profile:<30}: mean={np.mean(terminal_debts):.1f}%  median={np.median(terminal_debts):.1f}%')

print('Done.')

In [ ]:
# --------------------------------------------------------------------------
# Figure 5: Box plot of terminal debt by profile
# --------------------------------------------------------------------------

# Order profiles by economy type
profile_order = [
    'Advanced_Low', 'Advanced_Medium', 'Advanced_High',
    'Emerging_Market_Low', 'Emerging_Market_Medium', 'Emerging_Market_High',
    'Developing_Low', 'Developing_Medium', 'Developing_High',
]

data_ordered = [profile_results[p] for p in profile_order]
labels_short  = [p.replace('_', '\n') for p in profile_order]

# Colour by economy type
box_colours = (['#1f77b4'] * 3 +   # Advanced — blue
               ['#ff7f0e'] * 3 +   # Emerging Market — orange
               ['#2ca02c'] * 3)    # Developing — green

fig, ax = plt.subplots(figsize=(14, 6))

bp = ax.boxplot(
    data_ordered,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
)

for patch, colour in zip(bp['boxes'], box_colours):
    patch.set_facecolor(colour)
    patch.set_alpha(0.65)

ax.axhline(60,  color='green', linestyle='--', linewidth=1.2, alpha=0.7, label='Maastricht 60%')
ax.axhline(200, color='red',   linestyle='--', linewidth=1.2, alpha=0.7, label='Crisis threshold 200%')

ax.set_xticks(range(1, 10))
ax.set_xticklabels(labels_short, fontsize=9)
ax.set_ylabel('Terminal Debt-to-GDP (%)', fontsize=12)
ax.set_title(
    'Terminal Debt Distribution by Profile — Maintain Policy (100 Episodes Each)',
    fontsize=13, fontweight='bold'
)

# Economy type legend
legend_patches = [
    mpatches.Patch(color='#1f77b4', alpha=0.65, label='Advanced'),
    mpatches.Patch(color='#ff7f0e', alpha=0.65, label='Emerging Market'),
    mpatches.Patch(color='#2ca02c', alpha=0.65, label='Developing'),
    mpatches.Patch(color='none', label=''),  # spacer
]
from matplotlib.lines import Line2D
legend_lines = [
    Line2D([0], [0], color='green', linestyle='--', label='Maastricht 60%'),
    Line2D([0], [0], color='red',   linestyle='--', label='Crisis threshold 200%'),
]
ax.legend(handles=legend_patches[:3] + legend_lines, fontsize=9, loc='upper left')

ax.grid(True, axis='y', alpha=0.3)
ax.set_xlim(0.5, 9.5)

# Vertical separators between economy groups
for x in [3.5, 6.5]:
    ax.axvline(x, color='grey', linestyle=':', alpha=0.5)

plt.tight_layout()
for ext in ['png', 'svg']:
    plt.savefig(f'{FIGURES_DIR}/fig_env_profile_comparison.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig_env_profile_comparison')

---
## Summary

The environment is verified to:
- Instantiate correctly for all 9 calibrated profiles.
- Respond sensibly to different fiscal policies (austerity reduces debt, stimulus increases it).
- Generate meaningful climate events that affect the fiscal balance.
- Produce visually interpretable episode trajectories suitable for dissertation figures.

**Next step:** Notebook 07 — Benchmark Implementation and RL baseline training.